## 3.3 Synthetic Regression Data

Machine learning is all about extracting information from data. While we might not care intrinsically about the patterns we ourselves baked into an artificial data generating model, synthetic datasets are nevertheless useful for didactic purposes:

- Evaluating the properties of our learning algorithms.
- Confirming that our implementations work as expected.

For example, if we create data for which the correct parameters are known a priori, we can check that our model can in fact recover them.

In [7]:
%matplotlib inline
# display matplotlib inline
import random
import torch
from d2l import torch as d2l

### 3.3.1 Generating Synthetic Data

In [8]:
class SyntheticRegressionData(d2l.DataModule):  #@save
    """Synthetic data for linear regression."""
    def __init__(self, w, b, noise=0.01, num_train=1000, num_val=1000,
                 batch_size=32):
        super().__init__()
        self.save_hyperparameters()  # saves w, b, noise, num_train, num_val, batch_size as self attributes
        n = num_train + num_val
        self.X = torch.randn(n, len(w))          # random input features, shape (n, num_features)
        noise = torch.randn(n, 1) * noise         # gaussian noise scaled by noise level
        self.y = torch.matmul(self.X, w.reshape((-1, 1))) + b + noise  # y = Xw + b + noise

In [9]:
data = SyntheticRegressionData(w=torch.tensor([2, -3.4]), b=4.2)
print(vars(data).keys())
print('features:', data.X[0],'\nlabel:', data.y[0])

dict_keys(['hparams', 'root', 'num_workers', 'w', 'b', 'noise', 'num_train', 'num_val', 'batch_size', 'X', 'y'])
features: tensor([-0.0361,  1.1936]) 
label: tensor([0.0730])


### 3.3.2 Reading the Dataset

In [11]:
@d2l.add_to_class(SyntheticRegressionData)
def get_dataloader(self, train):
    if train:
        indices = list(range(0, self.num_train))          # use first num_train samples
        random.shuffle(indices)                            # shuffle for random order each epoch
    else:
        indices = list(range(self.num_train, self.num_train + self.num_val))  # use remaining samples for validation
    for i in range(0, len(indices), self.batch_size):
        batch_indices = torch.tensor(indices[i: i + self.batch_size])
        yield self.X[batch_indices], self.y[batch_indices]  # yield one batch of (X, y) at a time


In [12]:
X, y = next(iter(data.train_dataloader()))
print('X shape:', X.shape, '\ny shape:', y.shape)

X shape: torch.Size([32, 2]) 
y shape: torch.Size([32, 1])


### 3.3.3 Concise Implementation of the Data Loader

In [ ]:
@d2l.add_to_class(d2l.DataModule)  #@save
def get_tensorloader(self, tensors, train, indices=slice(0, None)):
    tensors = tuple(a[indices] for a in tensors)          # slice each tensor (X, y) to the requested range
    dataset = torch.utils.data.TensorDataset(*tensors)    # wrap into a PyTorch dataset
    return torch.utils.data.DataLoader(dataset, self.batch_size, shuffle=train)  # shuffle only during training

@d2l.add_to_class(SyntheticRegressionData)  #@save
def get_dataloader(self, train):
    i = slice(0, self.num_train) if train else slice(self.num_train, None)  # pick train or val slice
    return self.get_tensorloader((self.X, self.y), train, i)

In [ ]:
X, y = next(iter(data.train_dataloader()))
print('X shape:', X.shape, '\ny shape:', y.shape)